In [1]:
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from uuid import uuid4

/Users/mickaelriss/Code/rag-legislation-ia/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
DIRECTORY = "../data"
DB_PATH = "./chroma_langchain_db"

In [3]:
def load_documents() -> list[Document]:
    documents = []

    for root, dirs, files in os.walk(DIRECTORY):
        for file in files:
            if file.endswith(".pdf"):
                file_path = os.path.join(root, file)
                loader = PyPDFLoader(file_path=file_path)
                document = loader.load()
                documents.extend(document)

    return documents

In [4]:
def chunk_documents(documents: list[Document]) -> list[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )
    return text_splitter.split_documents(documents)

In [5]:
def create_vector_store():
    embeddings = OllamaEmbeddings(model="nomic-embed-text")

    vector_store = Chroma(
        collection_name="legislation_collection",
        persist_directory=DB_PATH,
        embedding_function=embeddings
    )
    
    if vector_store._collection.count() == 0:
        print("Database creation...")
        documents = load_documents()
        chunks = chunk_documents(documents)
        uuids = [str(uuid4()) for _ in range(len(chunks))]
        vector_store.add_documents(documents=chunks, ids=uuids)
    else: 
        print("Database loaded!")

    return vector_store

In [6]:
# documents = load_documents()
vector_store = create_vector_store()
print(vector_store)
# print(documents[0].page_content)
# pprint.pp(documents[0].metadata["title"])
# pprint.pp(documents[0].metadata["source"])

Database creation...


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
